# Fan — autocorrelation lengths and physics-bounded masks

Normal **fan** log-mel clips (`DCASE2020Task2LogMelDataset`), per-file standardized, shape `(n_mels, T)`.

**Temporal** autocovariance at frequency $f$, lag $\tau$ (with $\mu_f$ the temporal mean at that $f$):

$$R_{t}(f, \tau) = \frac{1}{T - \tau} \sum_{t=1}^{T-\tau} \left( S(f, t) - \mu_f \right) \left( S(f, t+\tau) - \mu_f \right)$$

**Frequency** autocovariance at time $t$, lag $\nu$ (with $\mu_t$ the spectral mean at that time):

$$R_{f}(\nu, t) = \frac{1}{F - \nu} \sum_{f=1}^{F-\nu} \left( S(f, t) - \mu_t \right) \left( S(f+\nu, t) - \mu_t \right)$$

We use $\rho = R/R(0)$ and the first lag where $\rho$ falls below `corr_threshold` (default $1/e$) to obtain **$\tau_c$** (time) and **$\nu_c$** (mel). Those are minimum scales for rectangular masks and for Perlin grid resolution via `rand_perlin_2d_np` (inverse correlation length along each axis).

Regime flags `is_stationary` / `is_transient` are **heuristics** on top of $(\tau_c, \nu_c)$ (not from the formulas); adjust thresholds if they mis-classify a machine type.

## 1. Paths and imports

In [ ]:
import math
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


def find_project_root() -> Path:
    p = Path.cwd().resolve()
    for _ in range(10):
        if (p / "src" / "data" / "dataset.py").is_file():
            return p
        if p.parent == p:
            break
        p = p.parent
    raise FileNotFoundError("Could not find project root (src/data/dataset.py).")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_PATH = Path("/mnt/ssd/LaCie/dcase2020-task2-dev-dataset")
if not DATA_PATH.exists():
    DATA_PATH = Path("/mnt/ssd/LaCie/dcase2020_task2/dcase2020_task2_dev_dataset")
if not DATA_PATH.exists():
    DATA_PATH = PROJECT_ROOT / "data" / "dcase2020-task2-dev-dataset"
if not DATA_PATH.exists():
    DATA_PATH = PROJECT_ROOT / "dataset" / "dcase2020_task2_dev_dataset"
if not DATA_PATH.exists():
    DATA_PATH = Path("../../data/dcase2020-task2-dev-dataset").resolve()

MACHINE_TYPE = "fan"
RNG = np.random.default_rng(0)

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"DATA_PATH: {DATA_PATH} (exists={DATA_PATH.exists()})")

## 2. Load fan normal spectrograms `(N, n_mels, T)`

In [ ]:
from src.data.dataset import DCASE2020Task2LogMelDataset

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Missing dataset root: {DATA_PATH}")

train_ds = DCASE2020Task2LogMelDataset(root=str(DATA_PATH), machine_type=MACHINE_TYPE)
raw = train_ds.data.numpy()
normal_specs = raw[:, 0, :, :].astype(np.float64)
N, n_mels, T = normal_specs.shape
print(f"normal_specs: shape={normal_specs.shape}, dtype={normal_specs.dtype}")

## 3. $R_t$, $R_f$, correlation lengths $\tau_c$, $\nu_c$, and `PhysicsDrivenMasker`

`PhysicsDrivenMasker` uses `src.utils.anomalies.perlin.rand_perlin_2d_np` with `res \propto (F/\nu_c,\; T/\tau_c)` so blob size tracks measured predictability.

In [ ]:
from src.utils.anomalies.perlin import rand_perlin_2d_np


def temporal_Rt_rho(S: np.ndarray, max_lag: int) -> tuple[np.ndarray, np.ndarray]:
    """R_t(f,tau) and rho=R_t/R_t(f,0) for one (F,T) spectrogram (lags 0..max_lag)."""
    F, Tloc = S.shape
    max_lag = min(max_lag, Tloc - 1)
    mu_f = S.mean(axis=1, keepdims=True)
    X = S - mu_f
    R = np.zeros((F, max_lag + 1), dtype=np.float64)
    for tau in range(max_lag + 1):
        if tau == 0:
            R[:, 0] = np.mean(X * X, axis=1)
        else:
            R[:, tau] = np.mean(X[:, : Tloc - tau] * X[:, tau:Tloc], axis=1)
    v0 = np.maximum(R[:, 0:1], 1e-12)
    rho = R / v0
    return R, rho


def frequency_Rf_rho(S: np.ndarray, max_nu: int) -> tuple[np.ndarray, np.ndarray]:
    """R_f(nu,t) and rho=R_f/R_f(t,0) for one (F,T) spectrogram (lags 0..max_nu per time t)."""
    F, Tloc = S.shape
    max_nu = min(max_nu, F - 1)
    mu_t = S.mean(axis=0, keepdims=True)
    Y = S - mu_t
    R = np.zeros((Tloc, max_nu + 1), dtype=np.float64)
    for nu in range(max_nu + 1):
        if nu == 0:
            R[:, 0] = np.mean(Y * Y, axis=0)
        else:
            R[:, nu] = np.mean(Y[: F - nu, :] * Y[nu:F, :], axis=0)
    v0 = np.maximum(R[:, 0:1], 1e-12)
    rho = R / v0
    return R, rho


def _first_lag_below_threshold(rho_row: np.ndarray, thresh: float, start: int = 1) -> int:
    for lag in range(start, rho_row.size):
        if rho_row[lag] < thresh:
            return lag
    return int(rho_row.size - 1)


def compute_correlation_lengths(
    normal_specs: np.ndarray,
    corr_threshold: float | None = None,
    max_lag_frac: float = 0.5,
) -> dict:
    """Pool normal clips; return tau_c, nu_c and mean rho curves."""
    thr = float(np.exp(-1.0)) if corr_threshold is None else float(corr_threshold)
    N, F, Tloc = normal_specs.shape
    max_nu = F - 1
    max_lag = max(1, int(Tloc * max_lag_frac))

    rho_f_sum = np.zeros((Tloc, max_nu + 1), dtype=np.float64)
    nu_nt: list[int] = []
    for n in range(N):
        _, rho_f = frequency_Rf_rho(normal_specs[n], max_nu)
        rho_f_sum += rho_f
        for t in range(Tloc):
            nu_nt.append(_first_lag_below_threshold(rho_f[t], thr))
    rho_f_mean = rho_f_sum / N
    nu_star_nt = np.asarray(nu_nt, dtype=np.int32)
    nu_c = max(1, int(np.median(nu_star_nt)))

    rho_t_sum = np.zeros((F, max_lag + 1), dtype=np.float64)
    tau_nf_list: list[int] = []
    for n in range(N):
        _, rho_t = temporal_Rt_rho(normal_specs[n], max_lag)
        rho_t_sum += rho_t
        for f in range(F):
            tau_nf_list.append(_first_lag_below_threshold(rho_t[f], thr))
    rho_t_mean = rho_t_sum / N
    tau_nf = np.asarray(tau_nf_list, dtype=np.int32).reshape(N, F)
    tau_per_f = np.median(tau_nf, axis=0).astype(np.int32)
    tau_c = max(1, int(np.median(tau_per_f)))

    return {
        "corr_threshold": thr,
        "tau_c": tau_c,
        "nu_c": nu_c,
        "rho_t_mean": rho_t_mean,
        "rho_f_mean": rho_f_mean,
        "rho_t_1d": rho_t_mean.mean(axis=0),
        "rho_f_1d": rho_f_mean.mean(axis=0),
        "nu_star_nt": nu_star_nt,
        "tau_nf": tau_nf,
        "tau_per_f": tau_per_f,
    }


class PhysicsDrivenMasker:
    """
    Rectangular masks with extents bounded by (tau_c, nu_c); optional Perlin branch
    with grid resolution ~(F/nu_c, T/tau_c) on (n_mels, T).
    """

    def __init__(
        self,
        n_mels: int,
        T: int,
        tau_c: int,
        nu_c: int,
        rng: np.random.Generator | None = None,
        stationary_tau_frac: float = 0.5,
        transient_nu_frac: float = 0.5,
        perlin_prob: float = 0.10,
    ):
        self.n_mels = int(n_mels)
        self.T = int(T)
        self.tau_c = max(1, int(tau_c))
        self.nu_c = max(1, int(nu_c))
        self.rng = rng or np.random.default_rng()
        self.perlin_prob = float(perlin_prob)
        self.is_stationary = self.tau_c > (self.T * stationary_tau_frac)
        self.is_transient = self.nu_c > (self.n_mels * transient_nu_frac)

    def _perlin_res(self) -> tuple[int, int]:
        """Mel axis res[0], time axis res[1] — large correlation -> few cells (stretched blobs)."""
        res_f = max(1, self.n_mels // self.nu_c)
        res_t = max(1, self.T // self.tau_c)
        return res_f, res_t

    def _generate_spectromorphic_mask(self) -> np.ndarray:
        mask = np.zeros((self.n_mels, self.T), dtype=np.float32)
        target_area = float(self.rng.uniform(0.3, 0.6) * self.n_mels * self.T)
        attempts = 0
        max_attempts = 500

        while mask.sum() < target_area and attempts < max_attempts:
            attempts += 1
            if self.is_stationary:
                bw = int(self.rng.integers(self.nu_c, min(self.n_mels, self.nu_c * 3) + 1))
                bw = min(bw, self.n_mels)
                start_f = int(self.rng.integers(0, max(1, self.n_mels - bw + 1)))
                mask[start_f : start_f + bw, :] = 1.0
            elif self.is_transient:
                t_len = int(self.rng.integers(self.tau_c, min(self.T, self.tau_c * 3) + 1))
                t_len = min(t_len, self.T)
                t_start = int(self.rng.integers(0, max(1, self.T - t_len + 1)))
                mask[:, t_start : t_start + t_len] = 1.0
            else:
                bw = int(self.rng.integers(self.nu_c, min(self.n_mels, self.nu_c * 2) + 1))
                bw = min(bw, self.n_mels)
                t_len = int(self.rng.integers(self.tau_c, min(self.T, self.tau_c * 2) + 1))
                t_len = min(t_len, self.T)
                start_f = int(self.rng.integers(0, max(1, self.n_mels - bw + 1)))
                t_start = int(self.rng.integers(0, max(1, self.T - t_len + 1)))
                mask[start_f : start_f + bw, t_start : t_start + t_len] = 1.0

        return mask

    def _generate_physics_bounded_perlin(self) -> np.ndarray:
        res_f, res_t = self._perlin_res()
        noise = rand_perlin_2d_np((self.n_mels, self.T), (res_f, res_t))
        lo, hi = float(noise.min()), float(noise.max())
        u = (noise - lo) / (hi - lo + 1e-8)
        thresh = float(self.rng.uniform(0.35, 0.55))
        return (u > thresh).astype(np.float32)

    def generate_mask(self) -> np.ndarray:
        if self.rng.random() < self.perlin_prob:
            return self._generate_physics_bounded_perlin()
        return self._generate_spectromorphic_mask()

## 4. Fan: $\tau_c$, $\nu_c$, and masker

In [ ]:
stats = compute_correlation_lengths(normal_specs)
tau_c, nu_c = stats["tau_c"], stats["nu_c"]
thr = stats["corr_threshold"]

print(f"corr_threshold = {thr:.6f} (default 1/e)")
print(f"tau_c (temporal correlation length, frames) = {tau_c}")
print(f"nu_c (frequency correlation length, mel bins) = {nu_c}")

masker = PhysicsDrivenMasker(n_mels, T, tau_c=tau_c, nu_c=nu_c, rng=RNG)
print(f"masker.is_stationary (tau > {0.5}*T) = {masker.is_stationary}")
print(f"masker.is_transient (nu > {0.5}*n_mels) = {masker.is_transient}")
print(f"Perlin grid res (mel, time) ~= {masker._perlin_res()}")

## 5. Mean $\rho$ curves and example masks

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(stats["rho_t_1d"], color="C0")
ax[0].axhline(thr, color="k", linestyle=":", alpha=0.6)
ax[0].axvline(tau_c, color="C3", linestyle="--", label=f"tau_c = {tau_c}")
ax[0].set_xlabel(r"lag $\tau$")
ax[0].set_ylabel(r"mean$_f\,\rho_t(\tau)$")
ax[0].set_title("Temporal (pooled)")
ax[0].legend()

ax[1].plot(stats["rho_f_1d"], color="C1")
ax[1].axhline(thr, color="k", linestyle=":", alpha=0.6)
ax[1].axvline(nu_c, color="C3", linestyle="--", label=f"nu_c = {nu_c}")
ax[1].set_xlabel(r"lag $\nu$")
ax[1].set_ylabel(r"mean$_t\,\rho_f(\nu)$")
ax[1].set_title("Frequency (pooled)")
ax[1].legend()
plt.tight_layout()
plt.show()

spec_idx = 0
S = normal_specs[spec_idx]
n_show = 4
fig, axes = plt.subplots(2, n_show, figsize=(3 * n_show, 5))
for k in range(n_show):
    m = masker.generate_mask()
    axes[0, k].imshow(S, aspect="auto", origin="lower", cmap="magma")
    axes[0, k].set_title(f"spec {spec_idx}" if k == 0 else "")
    axes[0, k].set_ylabel("mel")
    axes[1, k].imshow(m, aspect="auto", origin="lower", cmap="gray_r", vmin=0, vmax=1)
    axes[1, k].set_xlabel("time")
    axes[1, k].set_title("mask")
plt.suptitle("Log-mel (row 0) and sampled masks (row 1)")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(1, 3, figsize=(11, 3.2))
ax[0].imshow(S, aspect="auto", origin="lower", cmap="magma")
ax[0].set_title("S")
m = masker.generate_mask()
ax[1].imshow(m, aspect="auto", origin="lower", cmap="gray_r", vmin=0, vmax=1)
ax[1].set_title("mask")
ax[2].imshow(S * (1.0 - m), aspect="auto", origin="lower", cmap="magma")
ax[2].set_title(r"$S \odot (1-m)$")
for a in ax:
    a.set_xlabel("time")
    a.set_ylabel("mel")
plt.tight_layout()
plt.show()